# Symmetrizers

This notebook focuses on the special map `symmetrizer(DoubleBasis(Btarget, Bsource))`. In the most common case, `Bsource` is the full tensor-product basis and `Btarget` is a symmetry sector. Then the symmetrizer extracts the coordinates of the symmetry-adapted component of a full-space state.

We will also lift those sector coordinates back to the full Hilbert space to see what the projected state looks like.

In [ ]:
import Pkg
using LinearAlgebra

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
    include(joinpath(REPO_ROOT, "src", "EDKit.jl"))
    using .EDKit
else
    using EDKit
end


## Parity projection from the full Hilbert space

For a parity basis, `symmetrizer(DoubleBasis(Bpar, Bfull))` is the map from full-space coordinates to parity-sector coordinates. Its adjoint lifts a parity-sector vector back to the full Hilbert space.

In [ ]:
L = 4
Bfull = TensorBasis(L = L, base = 2)
Bpar = ParityBasis(L = L, p = 1, base = 2, threaded = false)
Tpar = DoubleBasis(Bpar, Bfull)
Ppar = symmetrizer(Tpar)

summary_parity = (
    full_dimension = size(Bfull, 1),
    parity_dimension = size(Bpar, 1),
    symmetrizer_size = size(Ppar),
)
summary_parity


## Project a random state and lift it back

The vector `Ppar * psi_full` lives in parity-sector coordinates. Applying `Ppar'` to that result lifts it back to the full space. The lifted state is the parity-even component of the original vector.

This is the cleanest way to think about the action: sector coordinates first, then optional reconstruction.

In [ ]:
psi_full = randn(ComplexF64, size(Bfull, 1)) |> normalize
psi_parity = Ppar * psi_full
psi_lifted = Ppar' * psi_parity

summary_projection = (
    sector_norm = norm(psi_parity),
    lifted_norm = norm(psi_lifted),
    idempotence_error = norm(Ppar' * (Ppar * psi_lifted) - psi_lifted),
)
summary_projection


## The same idea with an Abelian basis

Now we repeat the construction with a more general `AbelianBasis`: half filling, zero momentum, and even parity. The code looks almost the same, which is exactly the point of the common `DoubleBasis` interface.

In [ ]:
Babel = basis(L = L, N = 2, k = 0, p = 1, base = 2, threaded = false)
Tabel = DoubleBasis(Babel, Bfull)
Pabel = symmetrizer(Tabel)
psi_abel = Pabel * psi_full
psi_abel_lifted = Pabel' * psi_abel

summary_abelian = (
    abelian_dimension = size(Babel, 1),
    abelian_symmetrizer_size = size(Pabel),
    abelian_lifted_norm = norm(psi_abel_lifted),
)
summary_abelian


## Visualize the weight kept by the projection

A simple plot makes the geometry more intuitive: the norm of the sector coordinates tells us how much of the original state lies in that symmetry sector.

In [ ]:
weights = [norm(psi_parity)^2, norm(psi_abel)^2]
labels = ["parity-even", "N=2, k=0, p=+1"]

if isdefined(Main, :IJulia)
    using Plots
    bar(labels, weights; ylabel = "projected weight", label = nothing, title = "How much of the state survives each projection")
else
    @info "Skipping Plots outside IJulia; returning projection weights instead"
    (; labels = labels, weights = weights)
end


## Key takeaway

The symmetrizer is the basis-only map that turns a state in a less symmetric basis into coordinates in a more symmetric basis. Its adjoint lifts those coordinates back to the original Hilbert space. This is the right conceptual tool when you want projection without introducing any local operator data.